<a href="https://colab.research.google.com/github/Ravinduranasinghe/Audio-Downloader-and-Converter/blob/main/Audio_Downloader_and_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, re, time, subprocess, zipfile, shutil, json
from urllib.request import urlopen, Request
from urllib.error import URLError
from google.colab import drive, files

# ============================================================
#  CELL 1 — Setup
# ============================================================
def setup():
    print("🔧 Installing ffmpeg...")
    subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)
    ffmpeg = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    print(f"✅ {ffmpeg.stdout.splitlines()[0]}")
    print("✅ Setup complete.\n")

setup()

# ============================================================
#  CONFIGURATION — edit if needed
# ============================================================
GDRIVE_FOLDER = "AudioDownloads"      # folder name inside your Google Drive root
TEMP_DIR      = "/content/temp_work"  # scratch space inside Colab

# Supported audio extensions to convert
AUDIO_EXTENSIONS = {'.mp3', '.wav', '.flac', '.aac', '.ogg', '.opus',
                    '.m4a', '.wma', '.webm', '.ape', '.aiff', '.aif'}

# ============================================================
#  STEP 1 — Upload URLs .txt file
# ============================================================
print("How would you like to provide download URLs?")
print("  1 — Paste links directly")
print("  2 — Upload a .txt file")
print()
choice = input("Enter 1 or 2: ").strip()

urls = []

if choice == "1":
    print()
    print("Paste your URLs below, one per line.")
    print("When done, type END on a new line and press Enter.")
    print()
    while True:
        line = input()
        if line.strip().upper() == "END":
            break
        line = line.strip()
        if line and not line.startswith("#"):
            urls.append(line)

elif choice == "2":
    print()
    print("📂 Upload your .txt file containing one URL per line.")
    uploaded = files.upload()
    if not uploaded:
        raise Exception("No file uploaded.")
    url_file = list(uploaded.keys())[0]
    with open(url_file) as f:
        urls = [line.strip() for line in f if line.strip() and not line.startswith("#")]

else:
    raise Exception("Invalid choice. Please enter 1 or 2.")

if not urls:
    raise Exception("No URLs provided.")

print(f"\n✅ Found {len(urls)} URL(s):\n")
for u in urls:
    print(f"  {u}")
print()

# ============================================================
#  STEP 2 — Mount Google Drive
# ============================================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("✅ Drive already mounted.")

save_path    = f"/content/drive/MyDrive/{GDRIVE_FOLDER}"
archive_file = f"{save_path}/.download_archive.txt"
os.makedirs(save_path, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)
print(f"📁 Output folder: Drive/{GDRIVE_FOLDER}")

# Load existing archive
archived = set()
if os.path.exists(archive_file):
    with open(archive_file) as f:
        for line in f:
            parts = line.strip().split(None, 1)
            if len(parts) == 2 and parts[0] == 'cd':
                archived.add(parts[1])
            elif len(parts) == 2 and parts[0] == 'cd-complete':
                archived.add(f"cd-complete:{parts[1]}")
    completed_cds = sum(1 for k in archived if k.startswith('cd-complete:'))
    print(f"📋 Archive: {len(archived)} track(s) done, {completed_cds} CD(s) fully complete (will skip zip download)")
else:
    print("📋 No archive found — starting fresh.")
print()

def archive_add(cd_id, track_key):
    """Append a completed track to the archive file."""
    with open(archive_file, 'a', encoding='utf-8') as f:
        f.write(f"cd {cd_id}:{track_key}\n")

def archive_key(cd_id, track_key):
    return f"{cd_id}:{track_key}"

# ============================================================
#  HELPERS
# ============================================================
def fetch_track_titles(cd_id):
    """
    Fetch Sinhala track titles from waharaka.com/listen/CDxxx-NN pages.
    Returns a dict: { track_number(int): title(str) }
    Tries tracks 01–20; stops after 3 consecutive 404s.
    """
    titles = {}
    misses = 0
    track_num = 1
    print(f"  🌐 Fetching track titles from waharaka.com for {cd_id}...")

    while misses < 3:
        url = f"https://www.waharaka.com/listen/{cd_id}-{track_num:02d}"
        try:
            req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(req, timeout=10) as resp:
                raw = resp.read()
                # Detect encoding from HTTP Content-Type header
                charset = resp.headers.get_content_charset()
                if not charset:
                    # Sniff charset from the first 2 KB of the raw response
                    sniff = raw[:2048].decode('ascii', errors='ignore')
                    meta_match = re.search(r'charset=["\']?([\w-]+)', sniff)
                    charset = meta_match.group(1) if meta_match else 'latin-1'
                html = raw.decode(charset, errors='replace')

            # Extract og:title meta tag — most reliable
            match = re.search(r'meta-og:title:\s*(.+)', html)
            if not match:
                # Fallback: <title> tag
                match = re.search(r'<title>([^<]+)</title>', html)
            if match:
                title = match.group(1).strip()
                # Skip generic titles like "සංයුක්ත තැටිය"
                if cd_id.upper() in title or len(title) < 5:
                    misses += 1
                else:
                    titles[track_num] = title
                    print(f"    #{track_num:02d} → {title[:60]}")
                    misses = 0
            else:
                misses += 1
        except Exception:
            misses += 1
        track_num += 1
        time.sleep(0.3)  # polite delay

    print(f"  ✅ Found {len(titles)} track title(s).")
    return titles


def map_files_to_tracks(audio_files):
    """
    Map a sorted list of audio files to sequential track numbers (1-based).
    A1, A2, B1, B2, C1... become tracks 1, 2, 3, 4, 5... by sort order.
    Returns a dict: { filepath: track_number }
    """
    def sort_key(path):
        base = os.path.splitext(os.path.basename(path))[0].upper()
        match = re.match(r'^([A-H])(\d+)', base)
        if match:
            return (ord(match.group(1)), int(match.group(2)))
        return (0, base)
    sorted_files = sorted(audio_files, key=sort_key)
    return {path: idx + 1 for idx, path in enumerate(sorted_files)}


def sanitize_filename(name):
    """Remove illegal characters and trim to fit Linux 255-byte filename limit.
    Sinhala UTF-8 chars are 3 bytes each, so we trim encoded length not char count."""
    name = re.sub(r'[\\/*?:"<>|\x00-\x1f]', '_', name).strip()
    # Reserve 5 bytes for the .opus extension
    max_bytes = 250
    encoded = name.encode('utf-8')
    if len(encoded) > max_bytes:
        # Trim bytes then decode safely (avoid cutting mid-character)
        trimmed = encoded[:max_bytes]
        name = trimmed.decode('utf-8', errors='ignore').rstrip()
    return name


def download_file(url, dest_dir):
    """Download a file with wget, return local path."""
    filename = url.split('/')[-1].split('?')[0]
    dest     = os.path.join(dest_dir, filename)
    if os.path.exists(dest):
        print(f"  ⏭️  Already downloaded: {filename}")
        return dest
    print(f"  ⬇️  Downloading: {filename}")
    result = subprocess.run(
        ['wget', '-q', '--show-progress', '-O', dest, url],
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"wget failed for {url}")
    print(f"  ✅ Downloaded: {filename}")
    return dest


def extract_zip(zip_path, extract_dir):
    """Extract a zip file, return list of extracted file paths."""
    print(f"  📦 Extracting: {os.path.basename(zip_path)}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    extracted = []
    for root, _, fnames in os.walk(extract_dir):
        for fname in fnames:
            extracted.append(os.path.join(root, fname))
    print(f"  ✅ Extracted {len(extracted)} file(s)")
    return extracted


def convert_to_opus(src_path, dest_dir, output_name):
    """Convert an audio file to opus 48k with given output_name (no extension).
    If a file with the same name already exists, appends _2, _3, etc."""
    base_name = sanitize_filename(output_name)
    dest_path = os.path.join(dest_dir, base_name + '.opus')

    # If file exists, check if it was already archived (same run skip) or a real duplicate
    if os.path.exists(dest_path):
        # Find a unique name by appending _2, _3, ...
        counter = 2
        while os.path.exists(os.path.join(dest_dir, f"{base_name}_{counter}.opus")):
            counter += 1
        new_name  = f"{base_name}_{counter}"
        dest_path = os.path.join(dest_dir, new_name + '.opus')
        print(f"    ⚠️  Duplicate name — saving as: {new_name}.opus")

    print(f"    🎵 Converting → {output_name}.opus")
    result = subprocess.run([
        'ffmpeg', '-y',
        '-i', src_path,
        '-c:a', 'libopus',
        '-b:a', '48k',
        '-vbr', 'on',
        '-compression_level', '10',
        '-application', 'audio',
        dest_path
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print(f"    ⚠️  FFmpeg error:\n{result.stderr[-500:]}")
        return None

    size_kb = os.path.getsize(dest_path) // 1024
    print(f"    ✅ Done ({size_kb} KB)")
    return dest_path

# ============================================================
#  STEP 3 — Process each URL
# ============================================================
total_converted = 0

for i, url in enumerate(urls, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(urls)}] {url}")
    print('='*60)

    zip_name    = url.split('/')[-1].split('?')[0]
    folder_name = os.path.splitext(zip_name)[0]   # e.g. "CD003"
    cd_id       = folder_name.upper()              # e.g. "CD003"
    extract_dir = os.path.join(TEMP_DIR, folder_name)
    output_dir  = os.path.join(save_path, folder_name)

    os.makedirs(extract_dir, exist_ok=True)
    os.makedirs(output_dir,  exist_ok=True)

    try:
        # 1) Check if entire CD is already done by looking for a cd-level archive entry
        cd_done_key = f"cd-complete:{cd_id}"
        if cd_done_key in archived:
            print(f"  ⏭️  Entire CD already completed (archived) — skipping.")
            continue

        # 2) Fetch track titles from website
        track_titles = fetch_track_titles(cd_id)

        # 3) Download zip
        zip_path = download_file(url, TEMP_DIR)

        # 4) Extract
        extracted_files = extract_zip(zip_path, extract_dir)

        # 5) Separate audio files and map to track numbers sequentially
        audio_files = [
            f for f in extracted_files
            if os.path.splitext(f)[1].lower() in AUDIO_EXTENSIONS
        ]
        file_track_map = map_files_to_tracks(audio_files)

        print(f"\n  🎧 {len(audio_files)} audio file(s) to convert.")

        pending = {
            af: track_num for af, track_num in file_track_map.items()
            if archive_key(cd_id, os.path.splitext(os.path.basename(af))[0]) not in archived
        }
        skipped = len(file_track_map) - len(pending)
        if skipped:
            print(f"  ⏭️  Skipping {skipped} already archived track(s).")

        for af, track_num in pending.items():
            original_base = os.path.splitext(os.path.basename(af))[0]
            track_key     = original_base  # e.g. "A1-edit"

            if track_num in track_titles:
                output_name = track_titles[track_num]
                print(f"    🏷️  {original_base} → #{track_num:02d}: {output_name[:55]}")
            else:
                output_name = original_base
                print(f"    ℹ️  {original_base} → no title found, using original name")

            result = convert_to_opus(af, output_dir, output_name)
            if result:
                total_converted += 1
                archive_add(cd_id, track_key)
                archived.add(archive_key(cd_id, track_key))

        # 6) If all tracks done, write a CD-level completion entry to archive
        all_keys = {archive_key(cd_id, os.path.splitext(os.path.basename(af))[0]) for af in file_track_map}
        if all_keys.issubset(archived):
            with open(archive_file, 'a', encoding='utf-8') as f:
                f.write(f"cd-complete {cd_id}\n")
            archived.add(cd_done_key)
            print(f"  ✅ All tracks done — CD marked complete in archive.")

        # 7) Clean up temp files
        os.remove(zip_path)
        shutil.rmtree(extract_dir, ignore_errors=True)
        print(f"\n  🗑️  Temp files cleaned up.")

    except Exception as e:
        print(f"\n  ❌ Error processing {url}:\n     {e}")
        continue

# ============================================================
#  DONE
# ============================================================
print(f"\n{'='*60}")
print(f"✅ All done! {total_converted} file(s) converted to opus 48k.")
print(f"📁 Saved to: Drive/{GDRIVE_FOLDER}/")
print(f"📋 Archive: Drive/{GDRIVE_FOLDER}/.download_archive.txt ({len(archived)} total entries)")
print('='*60)
print()
print("─" * 60)
print("💡 HOW TO RE-RUN")
print("─" * 60)
print("Completed tracks are recorded in .download_archive.txt in")
print("your Drive folder. Re-runs skip them even if .opus files")
print("were deleted from Drive.")
print()
print("To force re-download a specific track:")
print("  1. Open .download_archive.txt in Drive")
print("  2. Delete the line:  cd CDxxx:A1-edit")
print("  3. Save and re-run")

🔧 Installing ffmpeg...
✅ ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
✅ Setup complete.

How would you like to provide download URLs?
  1 — Paste links directly
  2 — Upload a .txt file

Enter 1 or 2: 2

📂 Upload your .txt file containing one URL per line.


Saving Direct Links.txt to Direct Links (3).txt

✅ Found 22 URL(s):

  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Arunodaya.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Surya-Alokaya.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Swayan-Prabhaa-Aalokaya.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Thilakuna-Denagamu.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Karmasthana-Thulin-Niwana-Karaa.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Samma-Samadhi-Athwal-Bhawana.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Sadaham-Sakachchaa-01.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Sadaham-Sakachchaa-02.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Sadaham-Sakachchaa-03.zip
  https://storage.googleapis.com/www.waharaka.com/downloads/cds/Sadaham-Sakachchaa-04.zip
  https://storage.googleapis.com/ww